# Imports
---

In [ ]:
from pathlib import Path

import numpy as np
import scipy
import scipy.stats

import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

# Functions
---

## Data Loading Helpers

In [ ]:
def _read_forecast_csv(
    path: Path,
    tz: str,
) -> pd.DataFrame:
    """Read a forecast CSV exported with index=True from the forecasting notebook.

    Parameters
    ----------
    path : Path
        Path to the forecast CSV file.
    tz : str
        Target timezone for the index.

    Returns
    -------
    pd.DataFrame
        DataFrame with a timezone-aware DatetimeIndex named 'timestamp'.
    """
    df = pd.read_csv(path, index_col=0)
    df.index = pd.to_datetime(df.index, utc=True).tz_convert(tz)
    df.index.name = "timestamp"
    return df


def _enforce_eval_window(
    df: pd.DataFrame,
    start: pd.Timestamp,
    end: pd.Timestamp,
    skip_dates: set,
) -> pd.DataFrame:
    """Cut to the evaluation window and remove skipped dates."""
    df = df.loc[start:end].copy()
    if skip_dates:
        df = df.loc[~df.index.normalize().isin(skip_dates)]
    return df


def _model_view(
    df_quantile: pd.DataFrame,
    model_name: str,
    quantiles: list[float],
    y_true_col: str,
) -> pd.DataFrame:
    """Extract a single quantile model from df_quantile with standardised column names.

    Returns a DataFrame with y_true and q{tau:.3f} columns, as expected
    by all evaluation functions.
    """
    qcols_model = [f"{model_name}_q{q:.3f}" for q in quantiles]
    return pd.concat(
        [
            df_quantile[[y_true_col]],
            df_quantile[qcols_model].rename(
                columns={f"{model_name}_q{q:.3f}": f"q{q:.3f}" for q in quantiles}
            ),
        ],
        axis=1,
    )

## Data Validation Helpers

In [ ]:
def check_index(
    df: pd.DataFrame,
    name: str,
    start: pd.Timestamp,
    end: pd.Timestamp,
    freq: str,
    n_periods: int,
    skip_dates: set,
) -> None:
    """Verify the DatetimeIndex of a forecast DataFrame.

    Checks that the index covers the expected evaluation window exactly,
    forms a complete regular grid at the given frequency with no missing
    or extra timestamps, and that the total length is divisible by
    n_periods.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame to check.
    name : str
        Label used in error messages.
    start : pd.Timestamp
        Expected first timestamp.
    end : pd.Timestamp
        Expected last timestamp.
    freq : str
        Expected frequency, e.g. '15min'.
    n_periods : int
        Number of intraday periods, e.g. 96.
    skip_dates : set
        Dates excluded from the evaluation window.
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"[{name}] Index must be a DatetimeIndex.")
    if df.index.min() != start or df.index.max() != end:
        raise ValueError(
            f"[{name}] Index range mismatch: "
            f"min={df.index.min()}, max={df.index.max()} "
            f"vs expected {start}..{end}."
        )
    expected = pd.date_range(start=start, end=end, freq=freq, tz=start.tz)
    if skip_dates:
        expected = expected[~expected.normalize().isin(skip_dates)]
    missing = expected.difference(df.index)
    extra   = df.index.difference(expected)
    if len(missing) > 0 or len(extra) > 0:
        raise ValueError(
            f"[{name}] Index does not match the expected {freq} grid. "
            f"Missing={len(missing)}, Extra={len(extra)}. "
            f"First missing: {missing[:3].tolist() if len(missing) > 0 else None}."
        )
    if len(df) % n_periods != 0:
        raise ValueError(
            f"[{name}] len(df)={len(df)} is not divisible by n_periods={n_periods}."
        )


def check_missing(df: pd.DataFrame, name: str) -> None:
    """Raise ValueError if any column contains NaN values.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame to check.
    name : str
        Label used in error messages.
    """
    na_cols = df.columns[df.isna().any()].tolist()
    if na_cols:
        raise ValueError(f"[{name}] Missing values in columns: {na_cols}.")


def check_numeric(df: pd.DataFrame, cols: list[str], name: str) -> None:
    """Raise TypeError if any of the specified columns are non-numeric.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame to check.
    cols : list[str]
        Column names to check.
    name : str
        Label used in error messages.
    """
    non_numeric = [c for c in cols if not pd.api.types.is_numeric_dtype(df[c])]
    if non_numeric:
        raise TypeError(f"[{name}] Non-numeric columns: {non_numeric}.")


def check_quantile_monotonicity(
    df: pd.DataFrame,
    model_name: str,
    quantiles: list[float],
) -> None:
    """Raise ValueError if quantile columns are not monotonically non-decreasing.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing prefixed quantile columns.
    model_name : str
        Model key used to identify quantile columns.
    quantiles : list[float]
        Quantile levels in ascending order.
    """
    qcols = [f"{model_name}_q{q:.3f}" for q in quantiles]
    qmat  = df[qcols].to_numpy()
    if not np.all(np.diff(qmat, axis=1) >= -1e-12):
        bad_rows = np.where(np.any(np.diff(qmat, axis=1) < -1e-12, axis=1))[0]
        i = int(bad_rows[0])
        raise ValueError(
            f"[{model_name}] Quantiles not monotone at row {i}, "
            f"time={df.index[i]}, "
            f"values={df.iloc[i][qcols].to_dict()}."
        )

## **Point Forecasts**

### Metrics (MAE, RMSE)

In [ ]:
def mae_point(
    df: pd.DataFrame,
    model: str,
    y_true_col: str = "y_true",
) -> float:
    """Mean Absolute Error for a point forecast column.

    Parameters
    ----------
    df : pd.DataFrame
        Forecast DataFrame with model and y_true columns.
    model : str
        Column name of the point forecast.
    y_true_col : str
        Column name of realised prices. Defaults to 'y_true'.

    Returns
    -------
    float
        Mean absolute error.
    """
    if model not in df.columns:
        raise ValueError(f"Column '{model}' not found in DataFrame.")
    if y_true_col not in df.columns:
        raise ValueError(f"Column '{y_true_col}' not found in DataFrame.")
    return float(np.mean(np.abs(df[y_true_col].values - df[model].values)))


def rmse_point(
    df: pd.DataFrame,
    model: str,
    y_true_col: str = "y_true",
) -> float:
    """Root Mean Squared Error for a point forecast column.

    Parameters
    ----------
    df : pd.DataFrame
        Forecast DataFrame with model and y_true columns.
    model : str
        Column name of the point forecast.
    y_true_col : str
        Column name of realised prices. Defaults to 'y_true'.

    Returns
    -------
    float
        Root mean squared error.
    """
    if model not in df.columns:
        raise ValueError(f"Column '{model}' not found in DataFrame.")
    if y_true_col not in df.columns:
        raise ValueError(f"Column '{y_true_col}' not found in DataFrame.")
    return float(np.sqrt(np.mean((df[y_true_col].values - df[model].values) ** 2)))


def bias_point(
    df: pd.DataFrame,
    model: str,
    y_true_col: str = "y_true",
) -> float:
    """Mean forecast bias (signed error) for a point forecast column.

    Parameters
    ----------
    df : pd.DataFrame
        Forecast DataFrame with model and y_true columns.
    model : str
        Column name of the point forecast.
    y_true_col : str
        Column name of realised prices. Defaults to 'y_true'.

    Returns
    -------
    float
        Mean signed error (forecast minus realised).
    """
    if model not in df.columns:
        raise ValueError(f"Column '{model}' not found in DataFrame.")
    if y_true_col not in df.columns:
        raise ValueError(f"Column '{y_true_col}' not found in DataFrame.")
    return float(np.mean(df[model].values - df[y_true_col].values))

### GW Test (Absolute Error)

#### Calculation

In [ ]:
# -----------------------------------------------------------------------
# Giacomini-White (GW) Test for Conditional Predictive Accuracy (CPA)
#
# Based on: Lago et al. (2021), epftoolbox
# https://github.com/jeslago/epftoolbox
#
# Modifications relative to the original:
#   (1) TT = d.shape[0]   (was: np.max(d.shape))
#       np.max(d.shape) returns n_prices/day instead of n_days when
#       n_prices/day > n_days. This study uses 89 evaluation days and
#       96 MTUs, so the original is incorrect here.
#   (2) for h in range(H)  (was: for h in range(24))
#       The original is hardcoded for 24 hourly prices. This study uses
#       96 quarter-hourly MTUs.
# -----------------------------------------------------------------------
def GW(p_real, p_pred_1, p_pred_2, norm=1, version='multivariate'):
    """One-sided Giacomini-White test for Conditional Predictive Accuracy.

    Tests whether forecast p_pred_2 has significantly higher CPA than
    p_pred_1. The null hypothesis H0 states that the CPA of p_pred_1 is
    greater than or equal to that of p_pred_2. Rejecting H0 implies that
    p_pred_2 is significantly more accurate.

    Parameters
    ----------
    p_real : np.ndarray, shape (n_days, H)
        Realised market prices.
    p_pred_1 : np.ndarray, shape (n_days, H)
        Forecasts of model 1 (baseline).
    p_pred_2 : np.ndarray, shape (n_days, H)
        Forecasts of model 2 (challenger).
    norm : {1, 2}
        Loss norm used to compute the loss differential series.
    version : {'multivariate', 'univariate'}
        'multivariate' averages the loss differential across all H prices
        before testing. 'univariate' runs a separate test for each MTU.

    Returns
    -------
    float or np.ndarray
        p-value (float) for 'multivariate', array of shape (H,) for
        'univariate'.
    """
    if p_real.shape != p_pred_1.shape or p_real.shape != p_pred_2.shape:
        raise ValueError('All three arrays must have the same shape.')
    if len(p_real.shape) == 1 or (len(p_real.shape) == 2 and p_real.shape[1] == 1):
        raise ValueError('Arrays must have shape (n_days, H) with H > 1.')

    loss1 = p_real - p_pred_1
    loss2 = p_real - p_pred_2

    tau = 1  # Single-step-ahead forecasts only.

    d = np.abs(loss1) - np.abs(loss2) if norm == 1 else loss1**2 - loss2**2

    TT = d.shape[0]  # n_days (see modification (1) above)
    H  = d.shape[1]  # n_prices/day

    if version == 'univariate':
        GWstat = np.inf * np.ones(H)
        for h in range(H):  # see modification (2) above
            # Minimal instrument set: constant + one lag of the loss differential.
            instruments = np.stack([np.ones_like(d[:-tau, h]), d[:-tau, h]])
            dh = d[tau:, h]
            T  = TT - tau
            instruments = np.array(instruments, ndmin=2)
            reg = instruments * dh  # element-wise: E[z_{t-1} * Delta_t] = 0
            if tau == 1:
                betas   = np.linalg.lstsq(reg.T, np.ones(T), rcond=None)[0]
                err     = np.ones((T, 1)) - reg.T @ betas
                GWstat[h] = T * (1 - np.mean(err**2))
            else:
                raise NotImplementedError('Only single-step forecasts are supported.')

    elif version == 'multivariate':
        d = d.mean(axis=1)  # Average loss differential across all H prices.
        instruments = np.stack([np.ones_like(d[:-tau]), d[:-tau]])
        d   = d[tau:]
        T   = TT - tau
        instruments = np.array(instruments, ndmin=2)
        reg = instruments * d
        if tau == 1:
            betas  = np.linalg.lstsq(reg.T, np.ones(T), rcond=None)[0]
            err    = np.ones((T, 1)) - reg.T @ betas
            GWstat = T * (1 - np.mean(err**2))
        else:
            raise NotImplementedError('Only single-step forecasts are supported.')

    else:
        raise ValueError("version must be 'univariate' or 'multivariate'.")

    # One-sided test: sign adjustment ensures the statistic is positive only
    # when model 2 outperforms model 1 on average.
    GWstat *= np.sign(np.mean(d, axis=0))

    q    = reg.shape[0]
    pval = 1 - scipy.stats.chi2.cdf(GWstat, q)
    return pval

In [ ]:
def gw_test_point(
    df: pd.DataFrame,
    model_1: str,
    model_2: str,
    y_true_col: str = "y_true",
    n_periods: int = 96,
    norm: int = 1,
    version: str = "multivariate",
) -> float | np.ndarray:
    """Wrapper for the GW test applied to point forecasts.

    Reshapes the flat forecast DataFrame into the (n_days, n_periods) array
    format required by GW() and returns the p-value(s).

    Parameters
    ----------
    df : pd.DataFrame
        Forecast DataFrame with one row per MTU.
    model_1 : str
        Column name of the benchmark forecast.
    model_2 : str
        Column name of the challenger forecast.
    y_true_col : str
        Column name of realised prices. Defaults to 'y_true'.
    n_periods : int
        Number of intraday periods. Defaults to 96 (15-min MTUs).
    norm : {1, 2}
        Loss norm passed to GW(). Defaults to 1.
    version : {'multivariate', 'univariate'}
        Passed to GW(). Defaults to 'multivariate'.

    Returns
    -------
    float or np.ndarray
        p-value (float) for 'multivariate', array of shape (n_periods,) for
        'univariate'.
    """
    y  = df[y_true_col].values.reshape(-1, n_periods)
    f1 = df[model_1].values.reshape(-1, n_periods)
    f2 = df[model_2].values.reshape(-1, n_periods)
    return GW(p_real=y, p_pred_1=f1, p_pred_2=f2, norm=norm, version=version)


#### Plotting

In [ ]:
# -----------------------------------------------------------------------
# Pairwise multivariate GW test heatmap
#
# Based on: plot_multivariate_GW_test, Lago et al. (2021), epftoolbox
# https://github.com/jeslago/epftoolbox
#
# Modifications relative to the original:
#   (1) reshape(-1, 96) instead of reshape(-1, 24) for 15-min MTU resolution.
#   (2) Grey diagonal entries (wx marker) instead of black for visual clarity.
#   (3) Explicit grid lines via LineCollection for a cleaner heatmap appearance.
#   (4) Optional label_map to replace internal model keys with display labels.
#   (5) Configurable figsize, fontsize, colorbar fraction/pad, and PDF/PNG export.
# -----------------------------------------------------------------------
def plot_multivariate_GW_test(
    real_price,
    forecasts,
    norm=1,
    title='GW test',
    savefig=False,
    path='',
    figsize=(12, 9),
    fontsize=10,
    label_map=None,
    fraction=0.03,
    pad=0.01,
):
    """Plot pairwise multivariate GW test results as a heatmap.

    Green cells indicate a low p-value: the model on the x-axis significantly
    outperforms the model on the y-axis (H0 rejected at that significance level).

    Parameters
    ----------
    real_price : pd.DataFrame
        Realised prices, shape (n_days * 96,) or (n_days * 96, 1).
    forecasts : pd.DataFrame
        One column per model, same length as real_price.
    norm : {1, 2}
        Loss norm for the GW loss differential. Defaults to 1.
    title : str
        Plot title.
    savefig : bool
        If True, saves the figure as PDF at path + '.pdf'.
    path : str
        File path prefix for saved figures.
    figsize : tuple
        Figure size in inches.
    fontsize : int
        Font size for tick labels and colorbar.
    label_map : dict, optional
        Mapping from DataFrame column names to display labels.
    fraction : float
        Colorbar width relative to axes height.
    pad : float
        Padding between axes and colorbar.
    """
    # Pairwise GW p-values
    p_values = pd.DataFrame(index=forecasts.columns, columns=forecasts.columns)
    for model1 in forecasts.columns:
        for model2 in forecasts.columns:
            if model1 == model2:
                p_values.loc[model1, model2] = 1
            else:
                p_values.loc[model1, model2] = GW(
                    p_real=real_price.values.reshape(-1, 96),
                    p_pred_1=forecasts.loc[:, model1].values.reshape(-1, 96),
                    p_pred_2=forecasts.loc[:, model2].values.reshape(-1, 96),
                    norm=norm,
                    version='multivariate',
                )

    # Custom red-green colormap with grey diagonal marker
    diag_grey = 0.12  # RGB value used for the diagonal 'x' marker color
    red   = np.concatenate([np.linspace(0, 1, 50), np.linspace(1, 0.5, 50)[1:], [diag_grey]])
    green = np.concatenate([np.linspace(0.5, 1, 50), np.zeros(49), [diag_grey]])
    blue  = np.concatenate([np.zeros(99), [diag_grey]])
    rgb_color_map = mpl.colors.ListedColormap(
        np.stack([red, green, blue], axis=1)
    )

    tick_labels = [label_map.get(m, m) for m in forecasts.columns] if label_map else list(forecasts.columns)

    n = len(forecasts.columns)
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(
        p_values.astype(float).values,
        cmap=rgb_color_map,
        vmin=0,
        vmax=0.1,
        interpolation="nearest",
    )

    # Explicit grid lines
    lines = []
    for i in range(n + 1):
        lines.append([(i - 0.5, -0.5), (i - 0.5, n - 0.5)])
        lines.append([(-0.5, i - 0.5), (n - 0.5, i - 0.5)])
    ax.add_collection(LineCollection(lines, colors='black', linewidths=1.4, zorder=10))
    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(n - 0.5, -0.5)

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=fontsize)
    ax.set_yticklabels(tick_labels, fontsize=fontsize)
    ax.plot(range(n), range(n), 'wx', markersize=8, zorder=11)

    cbar = fig.colorbar(im, ax=ax, fraction=fraction, pad=pad)
    cbar.set_label("p-value", fontsize=fontsize + 1)
    cbar.ax.tick_params(labelsize=fontsize)

    if title:
        ax.set_title(title, fontsize=fontsize + 1)

    plt.tight_layout()
    if savefig:
        plt.savefig(path + '.pdf', dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
def plot_multivariate_GW_from_df_point(
    df_point: pd.DataFrame,
    model_cols: list[str],
    y_col: str = "y_true",
    n_periods: int = 96,
    norm: int = 1,
    title: str = "Multivariate GW test",
    figsize: tuple = (12, 9),
    fontsize: int = 10,
    label_map: dict = None,
    fraction: float = 0.03,
    pad: float = 0.01,
    savefig: bool = False,
    path: str = '',
):
    """Wrapper around plot_multivariate_GW_test for the df_point format.

    Extracts realised prices and model forecast columns from df_point
    and passes them to plot_multivariate_GW_test.

    Parameters
    ----------
    df_point : pd.DataFrame
        Forecast DataFrame with model columns and a y_true column.
    model_cols : list[str]
        Model column names to include in the comparison.
    y_col : str
        Column name of realised prices. Defaults to 'y_true'.
    n_periods : int
        Number of intraday periods. Defaults to 96 (15-min MTUs).
    norm : {1, 2}
        Loss norm for the GW loss differential. Defaults to 1.
    title : str
        Plot title.
    figsize : tuple
        Figure size in inches.
    fontsize : int
        Font size for tick labels and colorbar.
    label_map : dict, optional
        Mapping from column names to display labels.
    fraction : float
        Colorbar width relative to axes height.
    pad : float
        Padding between axes and colorbar.
    savefig : bool
        If True, saves the figure as PDF and PNG at path + '.pdf'/'.png'.
    path : str
        File path prefix for saved figures.
    """
    if len(df_point) % n_periods != 0:
        raise ValueError(f'len(df_point)={len(df_point)} is not divisible by n_periods={n_periods}.')
    plot_multivariate_GW_test(
        real_price=df_point[[y_col]],
        forecasts=df_point[model_cols],
        norm=norm,
        title=title,
        figsize=figsize,
        fontsize=fontsize,
        label_map=label_map,
        fraction=fraction,
        pad=pad,
        savefig=savefig,
        path=path,
    )

## **Quantile Forecasts**

### Metrics (MAE, RMSE, PS, APS, PICP)

In [ ]:
def mae_for_median(
    df: pd.DataFrame,
    y_true_col: str = "y_true",
    q_median: float = 0.5,
) -> float:
    """MAE of the median quantile forecast.

    The median minimises expected absolute loss, making MAE on q=0.5
    the natural point-forecast metric implied by quantile forecasts.

    Parameters
    ----------
    df : pd.DataFrame
        Forecast DataFrame with quantile columns and a y_true column.
    y_true_col : str
        Column name of realised prices. Defaults to 'y_true'.
    q_median : float
        Quantile level of the median forecast. Defaults to 0.5.

    Returns
    -------
    float
        Mean absolute error of the median quantile forecast.
    """
    qcol = f"q{q_median:.3f}"
    if y_true_col not in df.columns:
        raise ValueError(f"Missing column '{y_true_col}'.")
    if qcol not in df.columns:
        raise ValueError(f"Missing median quantile column '{qcol}'.")
    return np.mean(np.abs(df[y_true_col].to_numpy() - df[qcol].to_numpy()))

In [ ]:
def pinball_score(
    y: np.ndarray,
    q: np.ndarray,
    tau: float,
) -> np.ndarray:
    """Element-wise pinball (quantile) loss without aggregation.

    Parameters
    ----------
    y : np.ndarray
        Realised values.
    q : np.ndarray
        Quantile forecast values.
    tau : float
        Quantile level in (0, 1).

    Returns
    -------
    np.ndarray
        Element-wise pinball loss values.
    """
    diff = y - q
    return np.maximum(tau * diff, (tau - 1) * diff)


def aps_loss_matrix(
    df: pd.DataFrame,
    quantiles: list[float],
    y_true_col: str = "y_true",
    n_periods: int = 96,
) -> np.ndarray:
    """APS loss matrix of shape (n_days, n_periods).

    Each entry is the mean pinball loss across all quantile levels for
    one timestamp. The matrix format is required by the multivariate GW
    test, where rows correspond to days and columns to intraday MTUs.

    Parameters
    ----------
    df : pd.DataFrame
        Forecast DataFrame with quantile columns and a y_true column.
    quantiles : list[float]
        Quantile levels, e.g. [0.1, 0.25, 0.5, 0.75, 0.9].
    y_true_col : str
        Column name of realised prices. Defaults to 'y_true'.
    n_periods : int
        Number of intraday periods. Defaults to 96 (15-min MTUs).

    Returns
    -------
    np.ndarray
        APS loss matrix of shape (n_days, n_periods).
    """
    y      = df[y_true_col].to_numpy()
    aps_ts = sum(pinball_score(y, df[f"q{q:.3f}"].to_numpy(), q) for q in quantiles)
    aps_ts = aps_ts / len(quantiles)
    n_obs  = len(aps_ts)
    if n_obs % n_periods != 0:
        raise ValueError(f'len(df)={n_obs} is not divisible by n_periods={n_periods}.')
    return aps_ts.reshape(n_obs // n_periods, n_periods)


def aps_loss_per_timestamp(
    df: pd.DataFrame,
    quantiles: list[float],
    y_true_col: str = "y_true",
) -> pd.Series:
    """APS time series with one value per timestamp.

    Returns the mean pinball loss across all quantile levels for each
    timestamp, preserving the original index of df.

    Parameters
    ----------
    df : pd.DataFrame
        Forecast DataFrame with quantile columns and a y_true column.
    quantiles : list[float]
        Quantile levels, e.g. [0.1, 0.25, 0.5, 0.75, 0.9].
    y_true_col : str
        Column name of realised prices. Defaults to 'y_true'.

    Returns
    -------
    pd.Series
        APS time series of length n_obs with the original index.
    """
    y      = df[y_true_col].to_numpy()
    aps_ts = sum(pinball_score(y, df[f"q{q:.3f}"].to_numpy(), q) for q in quantiles)
    return pd.Series(aps_ts / len(quantiles), index=df.index, name="APS")


def aggregated_pinball_score(
    df: pd.DataFrame,
    quantiles: list[float],
    y_true_col: str = "y_true",
) -> float:
    """Aggregated Pinball Score (APS) as a scalar.

    Mean pinball loss averaged over all quantile levels and all
    timestamps. Primary probabilistic accuracy metric of this thesis,
    following Uniejewski and Weron (2021). Lower is better.

    Parameters
    ----------
    df : pd.DataFrame
        Forecast DataFrame with quantile columns and a y_true column.
    quantiles : list[float]
        Quantile levels, e.g. [0.1, 0.25, 0.5, 0.75, 0.9].
    y_true_col : str
        Column name of realised prices. Defaults to 'y_true'.

    Returns
    -------
    float
        Scalar APS value.
    """
    return aps_loss_per_timestamp(df, quantiles, y_true_col).mean()

In [ ]:
def coverage_indicator(
    df: pd.DataFrame,
    lower_q: float,
    upper_q: float,
    y_true_col: str = "y_true",
) -> pd.Series:
    """Binary indicator of prediction interval coverage.

    Returns 1 if the realised price falls within [q_lower, q_upper] at
    each timestamp, 0 otherwise. This series is the input to all
    coverage-based metrics.

    Parameters
    ----------
    df : pd.DataFrame
        Forecast DataFrame with quantile columns and a y_true column.
    lower_q : float
        Lower quantile level, e.g. 0.1.
    upper_q : float
        Upper quantile level, e.g. 0.9.
    y_true_col : str
        Column name of realised prices. Defaults to 'y_true'.

    Returns
    -------
    pd.Series
        Binary integer series of length n_obs.
    """
    lcol = f"q{lower_q:.3f}"
    ucol = f"q{upper_q:.3f}"
    if lcol not in df.columns or ucol not in df.columns:
        raise ValueError(f"Missing quantile column(s): '{lcol}', '{ucol}'.")
    y_true = df[y_true_col]
    q_lo   = df[lcol]
    q_hi   = df[ucol]
    return ((y_true >= q_lo) & (y_true <= q_hi)).astype(int)


def empirical_coverage(
    df: pd.DataFrame,
    lower_q: float,
    upper_q: float,
    y_true_col: str = "y_true",
) -> float:
    """Empirical coverage rate (PICP) of the prediction interval.

    Proportion of realised prices falling within [q_lower, q_upper].
    A well-calibrated model achieves empirical coverage close to the
    nominal level (upper_q - lower_q).

    Parameters
    ----------
    df : pd.DataFrame
        Forecast DataFrame with quantile columns and a y_true column.
    lower_q : float
        Lower quantile level, e.g. 0.1.
    upper_q : float
        Upper quantile level, e.g. 0.9.
    y_true_col : str
        Column name of realised prices. Defaults to 'y_true'.

    Returns
    -------
    float
        Empirical coverage rate in [0, 1].
    """
    return float(coverage_indicator(df, lower_q, upper_q, y_true_col).mean())

### Tests: Kupiec test, GW test (APS)

#### Kupiec Test

In [ ]:
def kupiec_test(
    indicators: np.ndarray,
    alpha: float,
) -> float:
    """Kupiec (1995) unconditional coverage (POF) test.

    Tests whether the empirical coverage rate is consistent with the
    nominal level alpha. H0: pi = alpha, where pi is the true coverage
    probability. Rejecting H0 implies significant miscoverage.

    Parameters
    ----------
    indicators : np.ndarray
        Binary coverage indicators (0/1).
    alpha : float
        Nominal coverage probability, e.g. 0.8 for an 80% PI.

    Returns
    -------
    float
        p-value of the likelihood-ratio test against chi2(1).
    """
    n     = len(indicators)
    k     = indicators.sum()
    if k == 0 or k == n:
        return 0.0
    pi_hat = k / n
    lr_stat = -2 * (
        (n - k) * np.log((1 - alpha) / (1 - pi_hat))
        + k * np.log(alpha / pi_hat)
    )
    return float(1 - scipy.stats.chi2.cdf(lr_stat, df=1))

In [ ]:
def mtu_kupiec_test(
    df: pd.DataFrame,
    q_low: float,
    q_high: float,
    alpha: float,
    y_true_col: str = "y_true",
    significance_level: float = 0.05,
) -> int:
    """Kupiec test applied separately to each of the 96 intraday MTUs.

    For each MTU the binary coverage indicators are extracted and the
    Kupiec unconditional coverage test is applied. Returns the number of
    MTUs for which H0 (correct unconditional coverage) is not rejected at
    the given significance level. A well-calibrated model should pass
    close to all 96 MTUs.

    Parameters
    ----------
    df : pd.DataFrame
        Forecast DataFrame with a 15-min DatetimeIndex, quantile columns,
        and a y_true column.
    q_low : float
        Lower quantile level, e.g. 0.1.
    q_high : float
        Upper quantile level, e.g. 0.9.
    alpha : float
        Nominal coverage probability, i.e. q_high - q_low.
    y_true_col : str
        Column name of realised prices. Defaults to 'y_true'.
    significance_level : float
        Significance level for the Kupiec test. Defaults to 0.05.

    Returns
    -------
    int
        Number of MTUs (out of 96) for which H0 is not rejected.
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("df.index must be a DatetimeIndex.")
    indicators = coverage_indicator(df, q_low, q_high, y_true_col).to_numpy()
    mtu_index  = df.index.hour * 4 + df.index.minute // 15 + 1
    return sum(
        kupiec_test(indicators[mtu_index == mtu], alpha) >= significance_level
        for mtu in range(1, 97)
        if (mtu_index == mtu).any()
    )

#### GW Test (APS)

##### Calculation

In [ ]:
# -----------------------------------------------------------------------
# GW test operating on precomputed loss matrices
#
# Adaptation of GW() (Lago et al., 2021) to support arbitrary loss
# functions such as APS. The original GW() computes the loss differential
# internally from raw price forecasts via L1/L2 norms. This version
# accepts precomputed loss matrices directly.
#
# Modifications relative to the original epftoolbox code:
#   (1) TT = d.shape[0] instead of np.max(d.shape): see GW() above.
#   (2) for h in range(n_periods) instead of range(24): see GW() above.
# -----------------------------------------------------------------------
def GW_loss(
    loss_1: np.ndarray,
    loss_2: np.ndarray,
    version: str = "multivariate",
) -> float | np.ndarray:
    """GW test for conditional predictive ability on precomputed loss matrices.

    H0: loss_1 has equal or better conditional predictive accuracy than
    loss_2. Rejecting H0 implies loss_2 is significantly more accurate.

    Parameters
    ----------
    loss_1 : np.ndarray, shape (n_days, n_periods)
        Loss matrix of the benchmark model.
    loss_2 : np.ndarray, shape (n_days, n_periods)
        Loss matrix of the challenger model.
    version : {'multivariate', 'univariate'}
        'multivariate' averages the loss differential across all intraday
        periods before testing (one scalar p-value). 'univariate' tests
        each period separately (array of n_periods p-values).

    Returns
    -------
    float or np.ndarray
        p-value for 'multivariate', array of shape (n_periods,) for
        'univariate'.
    """
    if loss_1.shape != loss_2.shape:
        raise ValueError("loss_1 and loss_2 must have the same shape.")
    if loss_1.ndim != 2:
        raise ValueError("Loss matrices must have shape (n_days, n_periods).")
    d = loss_1 - loss_2
    tau = 1
    n_days, n_periods = d.shape
    T = n_days - tau

    if version == "univariate":
        GWstat = np.full(n_periods, np.nan)
        for h in range(n_periods):
            dh          = d[tau:, h]
            instruments = np.vstack([np.ones(T), d[:-tau, h]])
            reg         = instruments * dh
            betas       = np.linalg.lstsq(reg.T, np.ones(T), rcond=None)[0]
            err         = np.ones(T) - reg.T @ betas
            GWstat[h]   = T * (1.0 - np.mean(err**2))
        GWstat *= np.sign(np.mean(d[tau:], axis=0))
        return 1.0 - scipy.stats.chi2.cdf(GWstat, df=2)

    elif version == "multivariate":
        d_bar       = d.mean(axis=1)
        dh          = d_bar[tau:]
        instruments = np.vstack([np.ones(T), d_bar[:-tau]])
        reg         = instruments * dh
        betas       = np.linalg.lstsq(reg.T, np.ones(T), rcond=None)[0]
        err         = np.ones(T) - reg.T @ betas
        GWstat      = T * (1.0 - np.mean(err**2))
        GWstat     *= np.sign(dh.mean())
        return 1.0 - scipy.stats.chi2.cdf(GWstat, df=2)

    else:
        raise ValueError("version must be 'univariate' or 'multivariate'.")

##### Plotting

In [ ]:
# -----------------------------------------------------------------------
# Pairwise multivariate GW test heatmap based on APS loss matrices
#
# Adaptation of plot_multivariate_GW_test (Lago et al., 2021) to operate
# on precomputed APS loss matrices via GW_loss() instead of raw price
# forecasts. The APS cannot be expressed as a simple L1/L2 norm on price
# forecast errors, so GW() is not directly applicable.
#
# Further modifications: see plot_multivariate_GW_test above.
# -----------------------------------------------------------------------
def plot_multivariate_GW_test_APS(
    loss_dict: dict,
    title: str = "GW test (APS)",
    savefig: bool = False,
    path: str = "",
    figsize: tuple = (10, 8),
    fontsize: int = 10,
    label_map: dict = None,
    fraction: float = 0.03,
    pad: float = 0.01,
) -> pd.DataFrame:
    """Pairwise multivariate GW test heatmap based on APS loss matrices.

    Green cells indicate a low p-value: the model on the x-axis
    significantly outperforms the model on the y-axis.

    Parameters
    ----------
    loss_dict : dict
        Mapping from model names to APS loss matrices of shape (n_days, n_periods).
    title : str
        Plot title. Pass an empty string to omit.
    savefig : bool
        If True, saves the figure as PDF at path + '.pdf'.
    path : str
        File path prefix for saved figures.
    figsize : tuple
        Figure size in inches.
    fontsize : int
        Font size for tick labels and colorbar.
    label_map : dict, optional
        Mapping from model keys to display labels.
    fraction : float
        Colorbar width relative to axes height.
    pad : float
        Padding between axes and colorbar.

    Returns
    -------
    pd.DataFrame
        Square DataFrame of pairwise p-values with model names as index and columns.
    """
    models = list(loss_dict.keys())
    n      = len(models)

    # Pairwise GW p-values
    p_values = pd.DataFrame(index=models, columns=models)
    for model_y in models:
        for model_x in models:
            if model_y == model_x:
                p_values.loc[model_y, model_x] = 1
            else:
                p_values.loc[model_y, model_x] = GW_loss(
                    loss_1=loss_dict[model_y],
                    loss_2=loss_dict[model_x],
                    version="multivariate",
                )

    # Custom red-green colormap with grey diagonal marker
    diag_grey = 0.12
    red   = np.concatenate([np.linspace(0, 1, 50), np.linspace(1, 0.5, 50)[1:], [diag_grey]])
    green = np.concatenate([np.linspace(0.5, 1, 50), np.zeros(49), [diag_grey]])
    blue  = np.concatenate([np.zeros(99), [diag_grey]])
    rgb_color_map = mpl.colors.ListedColormap(np.stack([red, green, blue], axis=1))

    tick_labels = [label_map.get(m, m) for m in models] if label_map is not None else list(models)

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(
        p_values.astype(float).values,
        cmap=rgb_color_map,
        vmin=0,
        vmax=0.1,
        interpolation="nearest",
    )

    # Explicit grid lines
    lines = []
    for i in range(n + 1):
        lines.append([(i - 0.5, -0.5), (i - 0.5, n - 0.5)])
        lines.append([(-0.5, i - 0.5), (n - 0.5, i - 0.5)])
    ax.add_collection(LineCollection(lines, colors='black', linewidths=1.4, zorder=10))
    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(n - 0.5, -0.5)

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=fontsize)
    ax.set_yticklabels(tick_labels, fontsize=fontsize)
    ax.plot(range(n), range(n), 'wx', markersize=8, zorder=11)

    cbar = fig.colorbar(im, ax=ax, fraction=fraction, pad=pad)
    cbar.set_label("p-value", fontsize=fontsize + 1)
    cbar.ax.tick_params(labelsize=fontsize)

    if title:
        ax.set_title(title, fontsize=fontsize + 1)

    plt.tight_layout()
    if savefig:
        plt.savefig(path + '.pdf', dpi=300, bbox_inches="tight")
    plt.show()
    return p_values

# Execution
---

## Settings

In [ ]:
# -----------------------------------------------------------------------
# Settings
# -----------------------------------------------------------------------

# Timezone for all timestamps
TZ = "Europe/Berlin"

# Evaluation window (inclusive)
TEST_START = pd.Timestamp("2025-12-01 00:00", tz=TZ)
TEST_END   = pd.Timestamp("2026-02-28 23:45", tz=TZ)

# Dates excluded from all metrics (e.g. missing data)
SKIP_DATES = {pd.Timestamp("2026-01-22", tz=TZ).normalize()}

# Market time resolution
FREQ      = "15min"
N_PERIODS = 96  # MTUs per day

# Quantile grid
QUANTILES = [0.10, 0.25, 0.50, 0.75, 0.90]
QCOLS     = [f"q{q:.3f}" for q in QUANTILES]

# Prediction intervals for coverage and Kupiec test: (lower_q, upper_q)
PREDICTION_INTERVALS = [
    (0.10, 0.90),  # 80% PI
    (0.25, 0.75),  # 50% PI
]

# Significance level for all statistical tests
SIGNIFICANCE_LEVEL = 0.05

# Column name for realised prices in all forecast DataFrames
Y_TRUE_COL = "y_true"

## Paths & File Registry

In [ ]:
# -----------------------------------------------------------------------
# Paths & File Registry
# -----------------------------------------------------------------------

try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path().resolve()

BASE_DIR    = NOTEBOOK_DIR.parent
RESULTS_DIR = BASE_DIR / "results"
OUTPUT_DIR  = BASE_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Point forecast paths ---

BASE_LEAR_ERA5      = RESULTS_DIR / "lear_op_results" / "era5"
BASE_LEAR_DWD       = RESULTS_DIR / "lear_op_results" / "dwd"
BASE_LEAR_EXAA_ONLY = RESULTS_DIR / "lear_op_results" / "exaa_only"

D_VALUES_ERA5 = [56, 112, 364]
D_VALUES_DWD  = [56]
C_VALUES      = [1, 5, 25]
VARIANTS      = ["fundamental", "exaa"]

POINT_MODEL_FILES = {}

for d in D_VALUES_ERA5:
    for c in C_VALUES:
        for v in VARIANTS:
            key  = f"era5_d{d}_c{c}_{v}"
            path = BASE_LEAR_ERA5 / f"d{d}" / f"c{c}" / v / "forecast.csv"
            POINT_MODEL_FILES[key] = path

for c in C_VALUES:
    for v in VARIANTS:
        key  = f"dwd_d56_c{c}_{v}"
        path = BASE_LEAR_DWD / "d56" / f"c{c}" / v / "forecast.csv"
        POINT_MODEL_FILES[key] = path

for d in [56, 112, 364]:
    key  = f"exaa_only_d{d}"
    path = BASE_LEAR_EXAA_ONLY / f"d{d}" / "forecast.csv"
    POINT_MODEL_FILES[key] = path

POINT_MODEL_FILES["exaa_naive"] = (
    RESULTS_DIR / "lear_op_results" / "exaa_naive" / "forecast.csv"
)

# --- Quantile forecast paths ---

BASE_SQRA = RESULTS_DIR / "sqra_results"

QUANTILE_MODEL_FILES = {
    "fund_ERA5"  : BASE_SQRA / "era5_fundamental"   / "forecast.csv",
    "fund_DWD"   : BASE_SQRA / "dwd_fundamental"    / "forecast.csv",
    "exaa_ERA5"  : BASE_SQRA / "era5_exaa_enriched" / "forecast.csv",
    "exaa_DWD"   : BASE_SQRA / "dwd_exaa_enriched"  / "forecast.csv",
    "exaa_naive" : BASE_SQRA / "exaa_naive"          / "forecast.csv",
    "exaa_only"  : BASE_SQRA / "exaa_only"           / "forecast.csv",
}

## Data Loading

In [ ]:
# -----------------------------------------------------------------------
# Data Loading
# -----------------------------------------------------------------------

y_true          = None
point_forecasts = []

for model_name, fp in POINT_MODEL_FILES.items():
    df_m = _enforce_eval_window(_read_forecast_csv(fp, TZ), TEST_START, TEST_END, SKIP_DATES)

    if y_true is None and Y_TRUE_COL in df_m.columns:
        y_true = df_m[[Y_TRUE_COL]]

    if "y_pred" not in df_m.columns:
        raise ValueError(f"[{model_name}] Missing 'y_pred' column in {fp}.")

    point_forecasts.append(df_m[["y_pred"]].rename(columns={"y_pred": model_name}))

if y_true is None:
    raise ValueError("Could not find y_true in any point forecast file.")

df_point = pd.concat([y_true] + point_forecasts, axis=1).sort_index()

# ---

y_true_q           = None
quantile_forecasts = []

for model_name, fp in QUANTILE_MODEL_FILES.items():
    df_m = _enforce_eval_window(_read_forecast_csv(fp, TZ), TEST_START, TEST_END, SKIP_DATES)

    if y_true_q is None and Y_TRUE_COL in df_m.columns:
        y_true_q = df_m[[Y_TRUE_COL]]

    missing = set(QCOLS) - set(df_m.columns)
    if missing:
        raise ValueError(f"[{model_name}] Missing quantile columns: {missing}.")

    quantile_forecasts.append(
        df_m[QCOLS].rename(columns={f"q{q:.3f}": f"{model_name}_q{q:.3f}" for q in QUANTILES})
    )

if y_true_q is None:
    raise ValueError("Could not find y_true in any quantile forecast file.")

df_quantile = pd.concat([y_true_q] + quantile_forecasts, axis=1).sort_index()

## Data Validation

In [ ]:
# -----------------------------------------------------------------------
# Data Validation
# -----------------------------------------------------------------------

# Index integrity
check_index(df_point,    "POINT",    TEST_START, TEST_END, FREQ, N_PERIODS, SKIP_DATES)
check_index(df_quantile, "QUANTILE", TEST_START, TEST_END, FREQ, N_PERIODS, SKIP_DATES)

# Missing values
check_missing(df_point,    "POINT")
check_missing(df_quantile, "QUANTILE")

# Numeric types
check_numeric(df_point,    df_point.columns.tolist(),    "POINT")
check_numeric(df_quantile, [Y_TRUE_COL],                 "QUANTILE")

# Quantile columns: numeric + monotonicity
for model_name in QUANTILE_MODEL_FILES:
    check_numeric(df_quantile, [f"{model_name}_q{q:.3f}" for q in QUANTILES], f"QUANTILE::{model_name}")
    check_quantile_monotonicity(df_quantile, model_name, QUANTILES)

print("All validation checks passed.")

## Point Forecast Evaluation

### Metrics

In [ ]:
# -----------------------------------------------------------------------
# Point Forecast Evaluation — Metrics
# -----------------------------------------------------------------------

POINT_MODELS = [c for c in df_point.columns if c != Y_TRUE_COL]

df_metrics_point = pd.DataFrame(
    {
        m: {
            "MAE":  mae_point(df_point, m, Y_TRUE_COL),
            "RMSE": rmse_point(df_point, m, Y_TRUE_COL),
            "Bias": bias_point(df_point, m, Y_TRUE_COL),
        }
        for m in POINT_MODELS
    }
).T.sort_values("MAE")

df_metrics_point

### GW Test

In [ ]:
# -----------------------------------------------------------------------
# Point Forecast Evaluation — GW Test
# -----------------------------------------------------------------------

MODEL_COLS_FUND = (
    [f"dwd_d56_c{c}_fundamental"   for c in C_VALUES]
    + [f"era5_d{d}_c{c}_fundamental" for d in D_VALUES_ERA5 for c in C_VALUES]
)

MODEL_COLS_EXAA = (
    ["exaa_naive"]
    + [f"dwd_d56_c{c}_exaa"   for c in C_VALUES]
    + [f"era5_d{d}_c{c}_exaa" for d in D_VALUES_ERA5 for c in C_VALUES]
    + [f"exaa_only_d{d}"      for d in [56, 112, 364]]
)

MODEL_COLS_ALL = MODEL_COLS_FUND + MODEL_COLS_EXAA

# Fundamental models (main text)
plot_multivariate_GW_from_df_point(
    df_point=df_point,
    model_cols=MODEL_COLS_FUND,
    y_col=Y_TRUE_COL,
    label_map=None,
    title="",
    figsize=(8, 7),
    fontsize=11,
    fraction=0.05,
    pad=0.02,
    savefig=True,
    path=str(OUTPUT_DIR / "gw_test_mae_fundamental"),
)

# EXAA models (main text)
plot_multivariate_GW_from_df_point(
    df_point=df_point,
    model_cols=MODEL_COLS_EXAA,
    y_col=Y_TRUE_COL,
    label_map=None,
    title="",
    figsize=(9, 8),
    fontsize=11,
    fraction=0.047,
    pad=0.02,
    savefig=True,
    path=str(OUTPUT_DIR / "gw_test_mae_exaa"),
)

# All models (Appendix)
plot_multivariate_GW_from_df_point(
    df_point=df_point,
    model_cols=MODEL_COLS_ALL,
    y_col=Y_TRUE_COL,
    label_map=None,
    title="",
    figsize=(13, 11),
    fontsize=9,
    fraction=0.03,
    pad=0.01,
    savefig=True,
    path=str(OUTPUT_DIR / "gw_test_mae_all"),
)

## Probabilistic Forecast Evaluation

### Model Setup

In [ ]:
# -----------------------------------------------------------------------
# Probabilistic Forecast Evaluation — Model Setup
# -----------------------------------------------------------------------

QUANTILE_MODELS = [
    "fund_DWD",
    "fund_ERA5",
    "exaa_naive",
    "exaa_DWD",
    "exaa_ERA5",
    "exaa_only",
]

LABEL_MAP_SQRA = {
    "fund_DWD"   : r"$\mathrm{SQRA}_{\mathrm{DWD,\,Fundamental}}$",
    "fund_ERA5"  : r"$\mathrm{SQRA}_{\mathrm{ERA5,\,Fundamental}}$",
    "exaa_naive" : r"$\mathrm{SQRA}_{\mathrm{EXAA,\,Naive}}$",
    "exaa_DWD"   : r"$\mathrm{SQRA}_{\mathrm{DWD,\,EXAA}}$",
    "exaa_ERA5"  : r"$\mathrm{SQRA}_{\mathrm{ERA5,\,EXAA}}$",
    "exaa_only"  : r"$\mathrm{SQRA}_{\mathrm{EXAA,\,Only}}$",
}

### Metrics

In [ ]:
# -----------------------------------------------------------------------
# Probabilistic Forecast Evaluation — Metrics
# -----------------------------------------------------------------------

# MAE (Median)
df_mae_median = pd.DataFrame(
    {
        model_name: {"MAE (Median)": mae_for_median(_model_view(df_quantile, model_name, QUANTILES, Y_TRUE_COL))}
        for model_name in QUANTILE_MODELS
    }
).T.sort_values("MAE (Median)")

df_mae_median

In [ ]:
# APS
df_aps = pd.DataFrame(
    {
        model_name: aps_loss_per_timestamp(
            _model_view(df_quantile, model_name, QUANTILES, Y_TRUE_COL),
            quantiles=QUANTILES,
            y_true_col=Y_TRUE_COL,
        )
        for model_name in QUANTILE_MODELS
    }
).sort_index()

df_aps_summary = (
    df_aps.mean()
    .rename("APS")
    .to_frame()
    .sort_values("APS")
)

df_aps_summary

### Coverage

In [ ]:
# -----------------------------------------------------------------------
# Probabilistic Forecast Evaluation — Coverage & Kupiec Test
# -----------------------------------------------------------------------

results_coverage = {}

for model_name in QUANTILE_MODELS:
    df_tmp = _model_view(df_quantile, model_name, QUANTILES, Y_TRUE_COL)
    model_results = {}

    for (q_low, q_high) in PREDICTION_INTERVALS:
        nominal    = q_high - q_low
        pi_label   = f"PI_{int(q_low*100)}_{int(q_high*100)}"

        model_results[pi_label] = {
            "Empirical Coverage"   : empirical_coverage(df_tmp, q_low, q_high, Y_TRUE_COL),
            "Nominal Coverage"     : nominal,
            "Kupiec MTUs passed"   : mtu_kupiec_test(
                                         df_tmp, q_low, q_high,
                                         alpha=nominal,
                                         y_true_col=Y_TRUE_COL,
                                         significance_level=SIGNIFICANCE_LEVEL,
                                     ),
        }

    results_coverage[model_name] = model_results

# Summary table per prediction interval
for (q_low, q_high) in PREDICTION_INTERVALS:
    pi_label = f"PI_{int(q_low*100)}_{int(q_high*100)}"
    df_cov = pd.DataFrame(
        {m: results_coverage[m][pi_label] for m in QUANTILE_MODELS}
    ).T
    print(pi_label)
    display(df_cov)

### GW Test

In [ ]:
# -----------------------------------------------------------------------
# Probabilistic Forecast Evaluation — GW Test (APS)
# -----------------------------------------------------------------------

loss_dict = {
    model_name: aps_loss_matrix(
        _model_view(df_quantile, model_name, QUANTILES, Y_TRUE_COL),
        quantiles=QUANTILES,
        y_true_col=Y_TRUE_COL,
        n_periods=N_PERIODS,
    )
    for model_name in QUANTILE_MODELS
}

_ = plot_multivariate_GW_test_APS(
    loss_dict=loss_dict,
    title="",
    figsize=(6, 4.5),
    fontsize=13,
    label_map=LABEL_MAP_SQRA,
    fraction=0.03,
    pad=0.01,
    savefig=True,
    path=str(OUTPUT_DIR / "gw_test_aps"),
)